In [2]:
# ============================================================
# TRANSFORMER ENCODER BASELINE
# Dynamic BdSL Recognition
# Student-Independent Evaluation
# 130 Features × 120 Frames
# ============================================================

import os
import time
import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    accuracy_score
)

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Embedding
)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

In [3]:



# ============================================================
# 1. SETTINGS
# ============================================================

BASE_DIR = r"D:\aaa EAAI Major Revision\dynamic\dynamic_dataset"

DATA_DIR = os.path.join(
    BASE_DIR,
    "BdSL_dynamic_npy"
)

TRAIN_PATH = os.path.join(
    DATA_DIR,
    "train_csv"
)

VAL_PATH = os.path.join(
    DATA_DIR,
    "validation_csv"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "test_csv"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results_transformer_baseline"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Dataset / Training Configuration
# ------------------------------------------------------------

SEQUENCE_LENGTH = 120
NUM_FEATURES = 130
EXPECTED_CLASSES = 40

BATCH_SIZE = 64
EPOCHS = 200
LEARNING_RATE = 0.0005


# ------------------------------------------------------------
# Transformer Configuration
# ------------------------------------------------------------

EMBED_DIM = 128

NUM_HEADS = 4

FF_DIM = 256

NUM_TRANSFORMER_BLOCKS = 2

TRANSFORMER_DROPOUT = 0.2


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

np.random.seed(42)
tf.random.set_seed(42)


# ============================================================
# 2. CHECK DATASET STRUCTURE
# ============================================================

print("=" * 70)
print("CHECKING DATASET")
print("=" * 70)

print("Train path:", TRAIN_PATH)
print("Validation path:", VAL_PATH)
print("Test path:", TEST_PATH)


if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        f"Train path not found: {TRAIN_PATH}"
    )

if not os.path.exists(VAL_PATH):
    raise FileNotFoundError(
        f"Validation path not found: {VAL_PATH}"
    )

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(
        f"Test path not found: {TEST_PATH}"
    )


# ============================================================
# 3. LOAD SEQUENCES
# ============================================================

def load_sequences_from_folder(base_path):

    class_names = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(
            os.path.join(base_path, d)
        )
    ])

    sequences = []
    labels = []

    print("\nLoading:", base_path)
    print("Classes found:", len(class_names))

    for class_name in class_names:

        class_path = os.path.join(
            base_path,
            class_name
        )

        video_names = sorted([
            d for d in os.listdir(class_path)
            if os.path.isdir(
                os.path.join(class_path, d)
            )
        ])

        print(
            f"{class_name:<35} "
            f"{len(video_names):>4} videos"
        )

        for video_name in video_names:

            video_path = os.path.join(
                class_path,
                video_name
            )

            frames = []
            valid_video = True

            for frame_num in range(
                1,
                SEQUENCE_LENGTH + 1
            ):

                frame_file = os.path.join(
                    video_path,
                    f"{frame_num:03d}.npy"
                )

                if not os.path.exists(frame_file):

                    print(
                        f"WARNING: Missing frame: "
                        f"{frame_file}"
                    )

                    valid_video = False
                    break

                frame = np.load(
                    frame_file
                )

                if frame.shape != (NUM_FEATURES,):

                    print(
                        f"WARNING: Wrong shape: "
                        f"{frame_file} -> {frame.shape}"
                    )

                    valid_video = False
                    break

                frames.append(
                    frame
                )

            if not valid_video:
                continue

            sequences.append(
                np.stack(frames)
            )

            labels.append(
                class_name
            )

    X = np.asarray(
        sequences,
        dtype=np.float32
    )

    y = np.asarray(
        labels
    )

    print("\nLoaded:")
    print("X shape:", X.shape)
    print("y shape:", y.shape)

    return X, y, class_names


# ============================================================
# 4. LOAD TRAIN / VALIDATION / TEST
# ============================================================

start_loading = time.time()


X_train, y_train, train_classes = (
    load_sequences_from_folder(
        TRAIN_PATH
    )
)

X_val, y_val, val_classes = (
    load_sequences_from_folder(
        VAL_PATH
    )
)

X_test, y_test, test_classes = (
    load_sequences_from_folder(
        TEST_PATH
    )
)


print(
    f"\nTotal loading time: "
    f"{time.time() - start_loading:.2f} seconds"
)


# ============================================================
# 5. DATA CHECKS
# ============================================================

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


if X_train.ndim != 3:
    raise ValueError(
        "X_train must be 3-dimensional."
    )

if X_train.shape[1] != SEQUENCE_LENGTH:
    raise ValueError(
        f"Expected {SEQUENCE_LENGTH} frames, "
        f"got {X_train.shape[1]}"
    )

if X_train.shape[2] != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} features, "
        f"got {X_train.shape[2]}"
    )

if len(train_classes) != EXPECTED_CLASSES:
    raise ValueError(
        f"Expected {EXPECTED_CLASSES} classes, "
        f"found {len(train_classes)}"
    )


# ============================================================
# 6. COMBINE CLASS NAMES
# ============================================================

actions = sorted(
    list(
        set(train_classes)
        | set(val_classes)
        | set(test_classes)
    )
)


print(
    "\nFinal classes:",
    len(actions)
)


if len(actions) != EXPECTED_CLASSES:
    raise ValueError(
        f"Expected {EXPECTED_CLASSES} total classes, "
        f"found {len(actions)}"
    )


# ============================================================
# 7. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

label_encoder.fit(
    actions
)


y_train_encoded = label_encoder.transform(
    y_train
)

y_val_encoded = label_encoder.transform(
    y_val
)

y_test_encoded = label_encoder.transform(
    y_test
)


with open(
    os.path.join(
        RESULTS_DIR,
        "label_encoder.pkl"
    ),
    "wb"
) as f:

    pickle.dump(
        label_encoder,
        f
    )


# ============================================================
# 8. STANDARDIZATION
# ============================================================

print("\n" + "=" * 70)
print("STANDARDIZATION")
print("=" * 70)


scaler = StandardScaler()


X_train_flat = X_train.reshape(
    -1,
    NUM_FEATURES
)

X_val_flat = X_val.reshape(
    -1,
    NUM_FEATURES
)

X_test_flat = X_test.reshape(
    -1,
    NUM_FEATURES
)


# Fit scaler ONLY on training data.

X_train_scaled = scaler.fit_transform(
    X_train_flat
).reshape(
    X_train.shape
)


X_val_scaled = scaler.transform(
    X_val_flat
).reshape(
    X_val.shape
)


X_test_scaled = scaler.transform(
    X_test_flat
).reshape(
    X_test.shape
)


with open(
    os.path.join(
        RESULTS_DIR,
        "scaler.pkl"
    ),
    "wb"
) as f:

    pickle.dump(
        scaler,
        f
    )


# ============================================================
# 9. ONE-HOT ENCODING
# ============================================================

y_train_cat = tf.keras.utils.to_categorical(
    y_train_encoded,
    num_classes=len(actions)
)

y_val_cat = tf.keras.utils.to_categorical(
    y_val_encoded,
    num_classes=len(actions)
)

y_test_cat = tf.keras.utils.to_categorical(
    y_test_encoded,
    num_classes=len(actions)
)


# ============================================================
# 10. TRANSFORMER ENCODER BLOCK
# ============================================================

def transformer_encoder(
    inputs,
    embed_dim,
    num_heads,
    ff_dim,
    dropout_rate
):

    # --------------------------------------------------------
    # Multi-Head Self-Attention
    # --------------------------------------------------------

    attention_output = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim // num_heads,
        dropout=dropout_rate
    )(
        inputs,
        inputs
    )

    attention_output = Dropout(
        dropout_rate
    )(
        attention_output
    )

    # Residual connection + normalization

    x = LayerNormalization(
        epsilon=1e-6
    )(
        inputs + attention_output
    )


    # --------------------------------------------------------
    # Feed-Forward Network
    # --------------------------------------------------------

    ffn = Dense(
        ff_dim,
        activation="relu"
    )(
        x
    )

    ffn = Dropout(
        dropout_rate
    )(
        ffn
    )

    ffn = Dense(
        embed_dim
    )(
        ffn
    )

    ffn = Dropout(
        dropout_rate
    )(
        ffn
    )


    # Residual connection + normalization

    output = LayerNormalization(
        epsilon=1e-6
    )(
        x + ffn
    )

    return output


# ============================================================
# 11. BUILD TRANSFORMER MODEL
# ============================================================

print("\n" + "=" * 70)
print("BUILDING TRANSFORMER ENCODER BASELINE")
print("=" * 70)


inp = Input(
    shape=(
        SEQUENCE_LENGTH,
        NUM_FEATURES
    )
)


# ------------------------------------------------------------
# Feature Projection
#
# Each frame initially contains 130 features.
# Project each frame to EMBED_DIM dimensions.
# ------------------------------------------------------------

x = Dense(
    EMBED_DIM
)(
    inp
)


# ------------------------------------------------------------
# Learnable Positional Embedding
#
# Transformer attention itself does not know temporal order.
# Therefore positional information is explicitly added.
# ------------------------------------------------------------

positions = tf.range(
    start=0,
    limit=SEQUENCE_LENGTH,
    delta=1
)


position_embedding_layer = Embedding(
    input_dim=SEQUENCE_LENGTH,
    output_dim=EMBED_DIM,
    name="positional_embedding"
)


position_embeddings = position_embedding_layer(
    positions
)


x = x + position_embeddings


# ------------------------------------------------------------
# Transformer Encoder Blocks
# ------------------------------------------------------------

for _ in range(
    NUM_TRANSFORMER_BLOCKS
):

    x = transformer_encoder(
        inputs=x,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM,
        dropout_rate=TRANSFORMER_DROPOUT
    )


# ------------------------------------------------------------
# Global Temporal Pooling
# ------------------------------------------------------------

x = GlobalAveragePooling1D()(x)


# ------------------------------------------------------------
# Classification Head
# ------------------------------------------------------------

x = Dense(
    128,
    activation="relu",
    kernel_regularizer=l2(1e-5)
)(x)

x = Dropout(
    0.3
)(x)


# ------------------------------------------------------------
# Output Layer
# ------------------------------------------------------------

out = Dense(
    len(actions),
    activation="softmax"
)(x)


model = Model(
    inputs=inp,
    outputs=out,
    name="Transformer_Encoder_Baseline"
)


# ============================================================
# 12. COMPILE MODEL
# ============================================================

optimizer = Adam(
    learning_rate=LEARNING_RATE
)


model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# 13. MODEL SUMMARY
# ============================================================

model.summary()


with open(
    os.path.join(
        RESULTS_DIR,
        "model_summary.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    model.summary(
        print_fn=lambda line:
        f.write(line + "\n")
    )


TOTAL_PARAMS = model.count_params()


print(
    f"\nTotal parameters: "
    f"{TOTAL_PARAMS:,}"
)


# ============================================================
# 14. CALLBACKS
# ============================================================

best_model_path = os.path.join(
    RESULTS_DIR,
    "best_transformer.keras"
)


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    mode="min",
    verbose=1
)


model_checkpoint = ModelCheckpoint(
    best_model_path,
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)


reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)


# ============================================================
# 15. TRAINING
# ============================================================

print("\n" + "=" * 70)
print("TRAINING TRANSFORMER")
print("=" * 70)


training_start = time.time()


history = model.fit(
    X_train_scaled,
    y_train_cat,

    validation_data=(
        X_val_scaled,
        y_val_cat
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=[
        early_stopping,
        model_checkpoint,
        reduce_lr
    ],

    verbose=1
)


training_time = (
    time.time()
    - training_start
)


print(
    f"\nTraining completed in "
    f"{training_time / 60:.2f} minutes"
)


# ============================================================
# 16. SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history.history
)

history_df.index.name = "epoch"


history_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "training_history.csv"
    )
)


# ============================================================
# 17. LOAD BEST MODEL
# ============================================================

best_model = tf.keras.models.load_model(
    best_model_path
)


# ============================================================
# 18. HELD-OUT TEST EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("HELD-OUT TEST SET EVALUATION")
print("=" * 70)


test_loss, test_accuracy = best_model.evaluate(
    X_test_scaled,
    y_test_cat,
    batch_size=BATCH_SIZE,
    verbose=1
)


# ============================================================
# 19. TEST PREDICTIONS
# ============================================================

y_pred_prob = best_model.predict(
    X_test_scaled,
    batch_size=BATCH_SIZE,
    verbose=1
)


y_pred = np.argmax(
    y_pred_prob,
    axis=1
)


y_true = y_test_encoded


# ============================================================
# 20. CLASSIFICATION REPORT
# ============================================================

class_names = list(
    label_encoder.classes_
)


report_dict = classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)


report_df = pd.DataFrame(
    report_dict
).transpose()


report_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "classification_report.csv"
    )
)


report_text = classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    digits=4,
    zero_division=0
)


print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    report_text
)


with open(
    os.path.join(
        RESULTS_DIR,
        "classification_report.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(
        report_text
    )


# ============================================================
# 21. EXTRACT TABLE METRICS
# ============================================================

prediction_accuracy = accuracy_score(
    y_true,
    y_pred
)


macro_precision = report_dict[
    "macro avg"
]["precision"]

macro_recall = report_dict[
    "macro avg"
]["recall"]

macro_f1 = report_dict[
    "macro avg"
]["f1-score"]


weighted_precision = report_dict[
    "weighted avg"
]["precision"]

weighted_recall = report_dict[
    "weighted avg"
]["recall"]

weighted_f1 = report_dict[
    "weighted avg"
]["f1-score"]


epochs_trained = len(
    history.history["loss"]
)


# ============================================================
# 22. SAVE SUMMARY FOR MANUSCRIPT TABLE
# ============================================================

summary_df = pd.DataFrame([{
    "Model": "Transformer Encoder",
    "Input Shape": f"{SEQUENCE_LENGTH}x{NUM_FEATURES}",
    "Parameters": TOTAL_PARAMS,
    "Epochs Trained": epochs_trained,
    "Test Loss": test_loss,
    "Test Accuracy": prediction_accuracy,
    "Macro Precision": macro_precision,
    "Macro Recall": macro_recall,
    "Macro F1": macro_f1,
    "Weighted Precision": weighted_precision,
    "Weighted Recall": weighted_recall,
    "Weighted F1": weighted_f1,
    "Training Time Minutes": training_time / 60
}])


summary_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "baseline_summary.csv"
    ),
    index=False
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TRANSFORMER BASELINE - FINAL RESULTS")
print("=" * 70)


print(
    f"Parameters         : "
    f"{TOTAL_PARAMS:,}"
)

print(
    f"Epochs trained     : "
    f"{epochs_trained}"
)

print(
    f"Test loss          : "
    f"{test_loss:.6f}"
)

print(
    f"Test accuracy      : "
    f"{prediction_accuracy * 100:.2f}%"
)

print(
    f"Macro precision    : "
    f"{macro_precision:.4f}"
)

print(
    f"Macro recall       : "
    f"{macro_recall:.4f}"
)

print(
    f"Macro F1-score     : "
    f"{macro_f1:.4f}"
)

print(
    f"Weighted precision : "
    f"{weighted_precision:.4f}"
)

print(
    f"Weighted recall    : "
    f"{weighted_recall:.4f}"
)

print(
    f"Weighted F1-score  : "
    f"{weighted_f1:.4f}"
)

print(
    f"Training time      : "
    f"{training_time / 60:.2f} minutes"
)

print(
    f"\nResults saved in:\n"
    f"{RESULTS_DIR}"
)

print("\nCompleted successfully.")

CHECKING DATASET
Train path: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\BdSL_dynamic_npy\train_csv
Validation path: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\BdSL_dynamic_npy\validation_csv
Test path: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\BdSL_dynamic_npy\test_csv

Loading: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\BdSL_dynamic_npy\train_csv
Classes found: 40
Good to See You                      174 videos
Leave                                150 videos
Please                               159 videos
What Time is It                      170 videos
Where Do You Live                    175 videos
abar dekha hbe                       162 videos
ami dukkhito                         161 videos
ami valo achi                        156 videos
apnake amar valo legeche             166 videos
apnake kivabe sahajjo korte pari     156 videos
apnar nam ki                         150 videos
apni kemon achen                     155 videos
apni ki kaj koren      

Model: "Transformer_Encoder_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 120, 130)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 120, 128)  │     16,768 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 120, 128)  │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 120, 128)  │     66,048 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 120, 128)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 120, 128)  │          0 │ add[0][0],        │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 120, 128)  │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 120, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 120, 256)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 120, 128)  │     32,896 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 120, 128)  │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 120, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 120, 128)  │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 120, 128)  │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 120, 128)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 120, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 120, 128)  │        256 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 120, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 120, 256)  │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 120, 128)  │     32,896 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 303,400 (1.16 MB)

 Trainable params: 303,400 (1.16 MB)

 Non-trainable params: 0 (0.00 B)


Total parameters: 303,400

TRAINING TRANSFORMER
Epoch 1/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step - accuracy: 0.3106 - loss: 2.5089
Epoch 1: val_loss improved from None to 1.13923, saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_transformer_baseline\best_transformer.keras

Epoch 1: finished saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_transformer_baseline\best_transformer.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 49s 382ms/step - accuracy: 0.3106 - loss: 2.5089 - val_accuracy: 0.7318 - val_loss: 1.1392 - learning_rate: 5.0000e-04
Epoch 2/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 336ms/step - accuracy: 0.6977 - loss: 1.1028
Epoch 2: val_loss improved from 1.13923 to 0.49597, saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_transformer_baseline\best_transformer.keras

Epoch 2: finished saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_transformer_baseline\best_transformer.keras
97/97 ━━━━━━━━━━